In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('Megamart examples').getOrCreate()

In [ ]:
data = spark.read.format("csv").option("header", True).option("inferschema", True).load("MegaMart.csv")

In [ ]:
data.show()

+--------+-------+----------+----------+----------------+--------------------+--------+--------------+--------------+------------+
|order_id|user_id|order_date|product_id|product_category|        product_name|quantity|price_per_unit|payment_method|order_status|
+--------+-------+----------+----------+----------------+--------------------+--------+--------------+--------------+------------+
|    1001|   U188|2025-04-20|      P940|         Fashion|            Sneakers|       2|         58.53|        PayPal|   Cancelled|
|    1002|   U062|2025-04-16|      P794|         Fashion|             T-Shirt|       3|         83.76|           UPI|    Returned|
|    1003|   U058|2025-04-18|      P326|         Fashion|          Sunglasses|       2|         78.85|        PayPal|  Processing|
|    1004|   U011|2025-04-10|      P574|         Fashion|          Sunglasses|       5|         46.49|        PayPal|   Delivered|
|    1005|   U003|2025-04-19|      P988|      Home Decor|         Photo Frame|     

In [ ]:
#Convert order_date from string to a DateType and extract year, month, day.
from pyspark.sql.functions import to_date, year, month, dayofmonth
data_datetransform = data.withColumn("order_date", to_date("order_date", "dd-mm-yyyy"))

data_year = data_datetransform.withColumn('year', year('order_date'))
data_month = data_datetransform.withColumn('month', month('order_date'))
data_day = data_month.withColumn("day", dayofmonth("order_date"))
data_day.show()

+--------+-------+----------+----------+----------------+--------------------+--------+--------------+--------------+------------+-----+---+
|order_id|user_id|order_date|product_id|product_category|        product_name|quantity|price_per_unit|payment_method|order_status|month|day|
+--------+-------+----------+----------+----------------+--------------------+--------+--------------+--------------+------------+-----+---+
|    1001|   U188|2025-04-20|      P940|         Fashion|            Sneakers|       2|         58.53|        PayPal|   Cancelled|    4| 20|
|    1002|   U062|2025-04-16|      P794|         Fashion|             T-Shirt|       3|         83.76|           UPI|    Returned|    4| 16|
|    1003|   U058|2025-04-18|      P326|         Fashion|          Sunglasses|       2|         78.85|        PayPal|  Processing|    4| 18|
|    1004|   U011|2025-04-10|      P574|         Fashion|          Sunglasses|       5|         46.49|        PayPal|   Delivered|    4| 10|
|    1005|   

In [ ]:
#Create a new column total_price = quantity * price_per_unit.
from pyspark.sql.functions import col
data_totalprice = data_day.withColumn("Total_price", col("quantity")*col("price_per_unit"))
data_totalprice.show()

+--------+-------+----------+----------+----------------+--------------------+--------+--------------+--------------+------------+-----+---+------------------+
|order_id|user_id|order_date|product_id|product_category|        product_name|quantity|price_per_unit|payment_method|order_status|month|day|       Total_price|
+--------+-------+----------+----------+----------------+--------------------+--------+--------------+--------------+------------+-----+---+------------------+
|    1001|   U188|2025-04-20|      P940|         Fashion|            Sneakers|       2|         58.53|        PayPal|   Cancelled|    4| 20|            117.06|
|    1002|   U062|2025-04-16|      P794|         Fashion|             T-Shirt|       3|         83.76|           UPI|    Returned|    4| 16|251.28000000000003|
|    1003|   U058|2025-04-18|      P326|         Fashion|          Sunglasses|       2|         78.85|        PayPal|  Processing|    4| 18|             157.7|
|    1004|   U011|2025-04-10|      P574|

In [ ]:
#Standardize categorical values (e.g., ensure order_status has consistent formatting like "Delivered" vs "delivered").
from pyspark.sql.functions import initcap
data_status = data_totalprice.withColumn("order_status", initcap("order_status"))
data_status.show()

+--------+-------+----------+----------+----------------+--------------------+--------+--------------+--------------+------------+-----+---+------------------+
|order_id|user_id|order_date|product_id|product_category|        product_name|quantity|price_per_unit|payment_method|order_status|month|day|       Total_price|
+--------+-------+----------+----------+----------------+--------------------+--------+--------------+--------------+------------+-----+---+------------------+
|    1001|   U188|2025-04-20|      P940|         Fashion|            Sneakers|       2|         58.53|        PayPal|   Cancelled|    4| 20|            117.06|
|    1002|   U062|2025-04-16|      P794|         Fashion|             T-Shirt|       3|         83.76|           UPI|    Returned|    4| 16|251.28000000000003|
|    1003|   U058|2025-04-18|      P326|         Fashion|          Sunglasses|       2|         78.85|        PayPal|  Processing|    4| 18|             157.7|
|    1004|   U011|2025-04-10|      P574|

In [ ]:
#Filter all orders where order_status = 'Delivered'.
data_delivered = data_status.filter(data_status.order_status == 'Delivered')
data_delivered.show()

+--------+-------+----------+----------+----------------+--------------------+--------+--------------+--------------+------------+-----+---+------------------+
|order_id|user_id|order_date|product_id|product_category|        product_name|quantity|price_per_unit|payment_method|order_status|month|day|       Total_price|
+--------+-------+----------+----------+----------------+--------------------+--------+--------------+--------------+------------+-----+---+------------------+
|    1004|   U011|2025-04-10|      P574|         Fashion|          Sunglasses|       5|         46.49|        PayPal|   Delivered|    4| 10|232.45000000000002|
|    1017|   U021|2025-04-26|      P728|           Books|  Big Data Explained|       5|         13.42|        PayPal|   Delivered|    4| 26|              67.1|
|    1019|   U115|2025-05-03|      P973|         Kitchen|             Blender|       3|         35.68|    Debit Card|   Delivered|    5|  3|107.03999999999999|
|    1022|   U027|2025-04-28|      P269|

In [ ]:
#Find the total revenue generated per product_category.
from pyspark.sql.functions import col
data_totalprice = data_day.withColumn("Total_price", col("quantity")*col("price_per_unit"))
data_grouped = data_totalprice.groupBy("product_category").sum("Total_price")
data_grouped.show()

+----------------+------------------+
|product_category|  sum(Total_price)|
+----------------+------------------+
|         Kitchen|32161.609999999986|
|         Fashion| 31437.23000000001|
|     Electronics|          34848.49|
|           Books|          31514.62|
|      Home Decor| 35400.27999999999|
+----------------+------------------+



In [ ]:
#Count the number of orders per payment method.

data_ordercounts = data_totalprice.groupby("payment_method").count()
data_ordercounts.show()

+--------------+-----+
|payment_method|count|
+--------------+-----+
|   Credit Card|  251|
|        PayPal|  261|
|    Debit Card|  224|
|           UPI|  264|
+--------------+-----+



In [ ]:
from pyspark.sql.functions import count
data_ordercounts1 = data_totalprice.groupby("payment_method").agg(count("order_id"))
data_ordercounts1.show()

+--------------+---------------+
|payment_method|count(order_id)|
+--------------+---------------+
|   Credit Card|            251|
|        PayPal|            261|
|    Debit Card|            224|
|           UPI|            264|
+--------------+---------------+



In [ ]:
#Find the top 5 products by sales (total_price)
from pyspark.sql.functions import sum,desc
data_top5 = data_totalprice.groupby("product_name").agg(sum('Total_price').alias("Revenue"))
data_top5.orderBy(desc("Revenue")).show()


+--------------------+------------------+
|        product_name|           Revenue|
+--------------------+------------------+
|          Wall Clock|           8241.02|
|             T-Shirt|7904.8200000000015|
|       Cushion Cover|           7834.53|
|          Smartwatch| 7750.909999999999|
|         USB-C Cable|            7586.4|
|            Wall Art| 7538.949999999999|
|   Spark for Dummies| 7216.649999999997|
|  Big Data Explained| 6917.890000000001|
|             Blender| 6802.619999999999|
|           Knife Set| 6752.650000000001|
|          Power Bank| 6655.379999999999|
|       Cutting Board| 6544.459999999998|
|         Photo Frame|6495.3099999999995|
|          Sunglasses| 6463.170000000002|
|      Wireless Mouse| 6432.240000000002|
|Bluetooth Headphones| 6423.559999999999|
|            Sneakers|           6412.27|
|Data Engineering 101| 6281.650000000001|
|           Microwave|           6278.89|
|             Toaster| 5782.990000000001|
+--------------------+------------

In [ ]:
#Calculate the average order value per user_id.
from pyspark.sql.functions import avg
data_aveordval = data_totalprice.groupBy("user_id").agg(avg("Total_price").alias("Average_order_value"))
data_aveordval.show()

+-------+-------------------+
|user_id|Average_order_value|
+-------+-------------------+
|   U185|            330.025|
|   U175| 126.36000000000001|
|   U014| 141.96666666666667|
|   U034|              134.5|
|   U200|            156.394|
|   U160| 135.62416666666667|
|   U180| 176.23909090909092|
|   U094|            205.212|
|   U067|             132.11|
|   U164|            139.425|
|   U127|             252.05|
|   U092| 112.73666666666666|
|   U039|          132.74625|
|   U176|            230.035|
|   U090| 181.19750000000002|
|   U053|           109.8775|
|   U133|            113.745|
|   U068|            273.975|
|   U080|             84.725|
|   U082|             391.86|
+-------+-------------------+
only showing top 20 rows



In [ ]:
#Find the category with maximum cancelled orders.
from pyspark.sql.functions import count,desc
data_cancelled = data_totalprice.filter(data_totalprice.order_status == 'Cancelled')
data_cancelled.groupBy("product_category").agg(count("order_status").alias("cancelled_orders_count")).orderBy(desc("cancelled_orders_count")).show()

+----------------+----------------------+
|product_category|cancelled_orders_count|
+----------------+----------------------+
|     Electronics|                    64|
|      Home Decor|                    54|
|         Kitchen|                    52|
|           Books|                    48|
|         Fashion|                    46|
+----------------+----------------------+



In [ ]:
from pyspark.sql.functions import col, sum, isnan, when
from pyspark.sql.types import DoubleType, FloatType

# Get the schema of the DataFrame
schema = data_totalprice.schema

# Prepare a list of sum expressions for null/NaN checks
null_nan_checks = []
for field in schema.fields:
    col_name = field.name
    col_type = field.dataType

    if isinstance(col_type, (DoubleType, FloatType)):
        # For DoubleType or FloatType, check for both null and NaN
        null_nan_checks.append(
            sum(when(col(col_name).isNull() | isnan(col(col_name)), 1).otherwise(0)).alias(col_name)
        )
    else:
        # For other types, only check for null
        null_nan_checks.append(
            sum(when(col(col_name).isNull(), 1).otherwise(0)).alias(col_name)
        )

# Execute the checks
data_totalprice.select(null_nan_checks).show()

+--------+-------+----------+----------+----------------+------------+--------+--------------+--------------+------------+-----+---+-----------+
|order_id|user_id|order_date|product_id|product_category|product_name|quantity|price_per_unit|payment_method|order_status|month|day|Total_price|
+--------+-------+----------+----------+----------------+------------+--------+--------------+--------------+------------+-----+---+-----------+
|       0|      0|         0|         0|               0|           0|       0|             0|             0|           0|    0|  0|          0|
+--------+-------+----------+----------+----------------+------------+--------+--------------+--------------+------------+-----+---+-----------+



In [ ]:
#Group orders by order_status and payment_method to see patterns (e.g., which payment mode has most cancellations).
from pyspark.sql.functions import asc

data_grouped_1 = data_totalprice.groupBy("order_status", "payment_method").agg(count("order_id").alias("order_count"))
data_grouped_1.orderBy(asc("order_status")).show()


+------------+--------------+-----------+
|order_status|payment_method|order_count|
+------------+--------------+-----------+
|   Cancelled|    Debit Card|         49|
|   Cancelled|           UPI|         75|
|   Cancelled|   Credit Card|         67|
|   Cancelled|        PayPal|         73|
|   Delivered|    Debit Card|         49|
|   Delivered|   Credit Card|         65|
|   Delivered|           UPI|         61|
|   Delivered|        PayPal|         63|
|  Processing|           UPI|         67|
|  Processing|    Debit Card|         71|
|  Processing|   Credit Card|         53|
|  Processing|        PayPal|         65|
|    Returned|    Debit Card|         55|
|    Returned|   Credit Card|         66|
|    Returned|           UPI|         61|
|    Returned|        PayPal|         60|
+------------+--------------+-----------+



In [ ]:
#Find users who purchased products from multiple categories.
#data_totalprice.filter(col("user_id") == 'U185').show()
from pyspark.sql.functions import countDistinct
data_user_multiple_categories = data_totalprice.groupBy("user_id").agg(countDistinct("product_category").alias("num_categories"))
data_user_multiple_categories.filter(col("num_categories") > 1).show()

+-------+--------------+
|user_id|num_categories|
+-------+--------------+
|   U185|             3|
|   U014|             4|
|   U034|             2|
|   U200|             3|
|   U160|             4|
|   U180|             4|
|   U094|             3|
|   U067|             3|
|   U164|             2|
|   U092|             4|
|   U127|             3|
|   U039|             4|
|   U176|             2|
|   U053|             2|
|   U090|             3|
|   U068|             2|
|   U133|             2|
|   U082|             2|
|   U107|             3|
|   U077|             2|
+-------+--------------+
only showing top 20 rows



In [ ]:
#Rank products within each product category by revenue.

from pyspark.sql.window import Window
from pyspark.sql.functions import rank, desc, col

window_spec = Window.partitionBy(["product_category", "product_name"]).orderBy(desc("Total_price"))
rnk = rank().over(window_spec)

data_totalprice.withColumn("rank", rnk).show()

+--------+-------+----------+----------+----------------+-------------+--------+--------------+--------------+------------+-----+---+------------------+----+
|order_id|user_id|order_date|product_id|product_category| product_name|quantity|price_per_unit|payment_method|order_status|month|day|       Total_price|rank|
+--------+-------+----------+----------+----------------+-------------+--------+--------------+--------------+------------+-----+---+------------------+----+
|    1298|   U138|2025-04-15|      P857|           Books|AI Revolution|       4|         94.67|        PayPal|    Returned|    4| 15|            378.68|   1|
|    1204|   U195|2025-04-18|      P913|           Books|AI Revolution|       5|         71.97|    Debit Card|    Returned|    4| 18|            359.85|   2|
|    1696|   U112|2025-04-10|      P784|           Books|AI Revolution|       5|         71.68|    Debit Card|   Cancelled|    4| 10|358.40000000000003|   3|
|    1210|   U085|2025-05-02|      P842|           B

In [ ]:
#For each user, calculate the running total spent over time.


#Find the latest order per user using row number or rank.


In [ ]:
#Calculate daily sales trend (group by order date).

from pyspark.sql.functions import sum, col

daily_sales_trend = data_totalprice.groupBy("order_date").agg(sum("Total_price").alias("Daily_Sales"))
daily_sales_trend.orderBy(col("order_date")).show()

+----------+------------------+
|order_date|       Daily_Sales|
+----------+------------------+
|2025-04-04| 7021.080000000001|
|2025-04-05| 8480.390000000003|
|2025-04-06|           7547.71|
|2025-04-07|           3494.92|
|2025-04-08|3553.7299999999996|
|2025-04-09|5717.3499999999985|
|2025-04-10|            5212.0|
|2025-04-11|           4617.83|
|2025-04-12|           4297.37|
|2025-04-13|           3384.15|
|2025-04-14|           5497.67|
|2025-04-15|           4943.78|
|2025-04-16| 6910.950000000001|
|2025-04-17|           3809.46|
|2025-04-18| 6823.180000000001|
|2025-04-19|           7357.66|
|2025-04-20| 4474.310000000001|
|2025-04-21| 7550.469999999999|
|2025-04-22| 6307.819999999998|
|2025-04-23|5258.9400000000005|
+----------+------------------+
only showing top 20 rows



In [ ]:
#Find the return rate = Returned Orders / Total Orders per category

from pyspark.sql.functions import col, count

# Calculate total orders per category
total_orders_per_category = data_totalprice.groupBy("product_category").agg(
    count("order_id").alias("total_orders")
)

# Calculate returned orders per category
returned_orders_per_category = data_totalprice.filter(col("order_status") == "Returned").groupBy("product_category").agg(
    count("order_id").alias("returned_orders")
)

# Join the two DataFrames and calculate the return rate
return_rate_per_category = total_orders_per_category.join(
    returned_orders_per_category,
    on="product_category",
    how="left"
).withColumn(
    "return_rate_percentage",
    (col("returned_orders") / col("total_orders")) * 100
).fillna(0) # Fill NaN with 0 for categories with no returns

return_rate_per_category.show()

+----------------+------------+---------------+----------------------+
|product_category|total_orders|returned_orders|return_rate_percentage|
+----------------+------------+---------------+----------------------+
|         Kitchen|         203|             55|    27.093596059113302|
|         Fashion|         188|             47|                  25.0|
|     Electronics|         223|             45|    20.179372197309416|
|           Books|         193|             48|    24.870466321243523|
|      Home Decor|         193|             47|    24.352331606217618|
+----------------+------------+---------------+----------------------+



In [ ]:
#Detect high-value customers (users who spent more than 5000 total).

from pyspark.sql.functions import sum, col

# Calculate total spending per user
total_spending_per_user = data_totalprice.groupBy("user_id").agg(sum("Total_price").alias("total_spent"))

# Filter for high-value customers (spent more than 5000)
high_value_customers = total_spending_per_user.filter(col("total_spent") > 5000)

high_value_customers.show()

+-------+-----------+
|user_id|total_spent|
+-------+-----------+
+-------+-----------+



In [ ]:
data_totalprice.groupb